# Train GMM trên RFM full-period

Notebook này huấn luyện Gaussian Mixture Model trên bảng `data/processed/rfm_uk_full.csv`.

Mục tiêu:

- Dùng bảng RFM full-period của United Kingdom.
- Giảm độ lệch của `Frequency` và `Monetary` bằng `log1p`.
- Chuẩn hóa dữ liệu bằng `StandardScaler`.
- So sánh số component GMM bằng AIC và BIC.
- Train GMM 10 cụm kỹ thuật, sau đó gộp thành 3 nhóm kinh doanh.
- Lưu model, scaler, profile cụm và action plan để dùng cho app Streamlit.

## 1. Load dữ liệu

Đoạn code dưới đây đọc bảng RFM đã được xây dựng từ toàn bộ dữ liệu giao dịch UK. Mỗi dòng tương ứng một khách hàng.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, MinMaxScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RFM_PATH = PROJECT_ROOT / "data" / "processed" / "rfm_uk_full.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

rfm = pd.read_csv(RFM_PATH)
rfm.head()

## 2. Tiền xử lý RFM

GMM được train trên ba biến `Recency`, `Frequency`, `Monetary`. Trong bản GMM này, `Recency` được giữ nguyên, còn `Frequency` và `Monetary` được biến đổi bằng `log1p` để giảm độ lệch phải trước khi chuẩn hóa.

In [ ]:
FEATURE_COLUMNS = ["Recency", "Frequency", "Monetary"]

train_data = rfm[FEATURE_COLUMNS].copy()
train_data["Frequency"] = np.log1p(train_data["Frequency"])
train_data["Monetary"] = np.log1p(train_data["Monetary"])

scaler = StandardScaler()
train_data_scaled = scaler.fit_transform(train_data)

pd.DataFrame(train_data_scaled, columns=FEATURE_COLUMNS).describe().round(3)

## 3. Chọn số component bằng AIC và BIC

Với GMM, AIC và BIC thường được dùng để so sánh số component. Giá trị càng thấp cho thấy mô hình mô tả dữ liệu tốt hơn sau khi đã tính đến độ phức tạp.

In [ ]:
RANDOM_STATE = 42

aic_scores = []
bic_scores = []
ks = range(2, 11)

for k in ks:
    gmm_test = GaussianMixture(n_components=k, random_state=RANDOM_STATE)
    gmm_test.fit(train_data_scaled)
    aic_scores.append(gmm_test.aic(train_data_scaled))
    bic_scores.append(gmm_test.bic(train_data_scaled))

score_df = pd.DataFrame({"K": list(ks), "AIC": aic_scores, "BIC": bic_scores})
score_df.round(2)

Biểu đồ dưới đây giúp nhìn xu hướng AIC/BIC khi tăng số component từ 2 đến 10.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(score_df["K"], score_df["AIC"], marker="o", label="AIC")
plt.plot(score_df["K"], score_df["BIC"], marker="o", label="BIC")
plt.xlabel("Số component GMM")
plt.ylabel("Giá trị AIC/BIC")
plt.title("So sánh AIC và BIC theo số component")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

best_k_aic = int(score_df.loc[score_df["AIC"].idxmin(), "K"])
best_k_bic = int(score_df.loc[score_df["BIC"].idxmin(), "K"])
print("K tốt nhất theo AIC:", best_k_aic)
print("K tốt nhất theo BIC:", best_k_bic)

## 4. Train GMM 10 cụm kỹ thuật

Trong phạm vi thử nghiệm, AIC và BIC thấp nhất tại K=10. Vì vậy mô hình được train với 10 cụm kỹ thuật. Các cụm kỹ thuật này sẽ được gộp thành 3 nhóm kinh doanh để dễ diễn giải trong báo cáo và app.

In [ ]:
N_COMPONENTS = 10

gmm = GaussianMixture(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
labels = gmm.fit_predict(train_data_scaled)

rfm_segmented = rfm.copy()
rfm_segmented["GMM_Cluster"] = labels
rfm_segmented.head()

## 5. Phân tích 10 cụm GMM

Bảng dưới đây cho biết quy mô, Recency, Frequency và Monetary trung bình/median của từng cụm kỹ thuật.

In [ ]:
technical_profile = (
    rfm_segmented.groupby("GMM_Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        RecencyMean=("Recency", "mean"),
        RecencyMedian=("Recency", "median"),
        FrequencyMean=("Frequency", "mean"),
        FrequencyMedian=("Frequency", "median"),
        MonetaryMean=("Monetary", "mean"),
        MonetaryMedian=("Monetary", "median"),
    )
    .reset_index()
)
technical_profile["CustomerPct"] = technical_profile["Customers"] / technical_profile["Customers"].sum() * 100
technical_profile["RevenuePct"] = technical_profile["Revenue"] / technical_profile["Revenue"].sum() * 100
technical_profile.round(2)

Heatmap dưới đây chuẩn hóa ba chỉ số RFM để so sánh tương đối giữa 10 cụm. Riêng Recency được đảo chiều vì Recency càng thấp càng tốt.

In [ ]:
heatmap_data = technical_profile[["RecencyMean", "FrequencyMean", "MonetaryMean"]].copy()
heatmap_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(heatmap_data),
    columns=["Recency", "Frequency", "Monetary"],
    index=technical_profile["GMM_Cluster"],
)
heatmap_scaled["Recency"] = 1 - heatmap_scaled["Recency"]

fig, ax = plt.subplots(figsize=(9, 6))
image = ax.imshow(heatmap_scaled.values, aspect="auto")
ax.set_xticks(range(3))
ax.set_xticklabels(["Recency", "Frequency", "Monetary"])
ax.set_yticks(range(len(heatmap_scaled)))
ax.set_yticklabels([f"Cụm {i}" for i in heatmap_scaled.index])
for i in range(len(heatmap_scaled)):
    for j in range(3):
        ax.text(j, i, f"{heatmap_scaled.iloc[i, j]:.2f}", ha="center", va="center")
plt.colorbar(image, label="Mức độ tốt sau chuẩn hóa")
plt.title("Heatmap đặc điểm RFM của 10 cụm GMM")
plt.show()

## 6. Gộp 10 cụm thành 3 nhóm kinh doanh

Dựa trên profile cụm, 10 cụm kỹ thuật được gộp thành 3 nhóm:

- Cụm 3: Khách hàng VIP.
- Cụm 4, 6, 8: Khách hàng tiềm năng.
- Các cụm 0, 1, 2, 5, 7, 9: Khách hàng bình thường.

In [ ]:
CLUSTER_MAPPING = {
    0: "Khách hàng bình thường",
    1: "Khách hàng bình thường",
    2: "Khách hàng bình thường",
    3: "Khách hàng VIP",
    4: "Khách hàng tiềm năng",
    5: "Khách hàng bình thường",
    6: "Khách hàng tiềm năng",
    7: "Khách hàng bình thường",
    8: "Khách hàng tiềm năng",
    9: "Khách hàng bình thường",
}

rfm_segmented["Business_Group"] = rfm_segmented["GMM_Cluster"].map(CLUSTER_MAPPING)

business_profile = (
    rfm_segmented.groupby("Business_Group")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        RecencyMean=("Recency", "mean"),
        RecencyMedian=("Recency", "median"),
        FrequencyMean=("Frequency", "mean"),
        FrequencyMedian=("Frequency", "median"),
        MonetaryMean=("Monetary", "mean"),
        MonetaryMedian=("Monetary", "median"),
    )
    .reset_index()
)
business_profile["CustomerPct"] = business_profile["Customers"] / business_profile["Customers"].sum() * 100
business_profile["RevenuePct"] = business_profile["Revenue"] / business_profile["Revenue"].sum() * 100
business_profile.round(2)

## 7. Lưu model và kết quả

Đoạn code cuối cùng lưu scaler, GMM, bảng map cụm, metadata và các bảng kết quả để app Streamlit sử dụng.

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_log_likelihood = gmm.score_samples(train_data_scaled)
ood_threshold = float(np.percentile(train_log_likelihood, 1))
average_order_value = rfm_segmented["Monetary"] / rfm_segmented["Frequency"]
input_limits = {
    "Recency_min": float(rfm_segmented["Recency"].quantile(0.01)),
    "Recency_max": float(rfm_segmented["Recency"].quantile(0.99)),
    "Frequency_min": float(rfm_segmented["Frequency"].quantile(0.01)),
    "Frequency_max": float(rfm_segmented["Frequency"].quantile(0.99)),
    "Monetary_min": float(rfm_segmented["Monetary"].quantile(0.01)),
    "Monetary_max": float(rfm_segmented["Monetary"].quantile(0.99)),
    "AverageOrderValue_min": float(average_order_value.quantile(0.01)),
    "AverageOrderValue_max": float(average_order_value.quantile(0.99)),
}

rfm_segmented.to_csv(PROCESSED_DIR / "rfm_gmm_segmented.csv", index=False)
technical_profile.round(4).to_csv(PROCESSED_DIR / "gmm_technical_cluster_profile.csv", index=False)
business_profile.round(4).to_csv(PROCESSED_DIR / "gmm_business_group_profile.csv", index=False)

joblib.dump(scaler, MODEL_DIR / "scaler_gmm.pkl")
joblib.dump(gmm, MODEL_DIR / "gmm.pkl")
joblib.dump(CLUSTER_MAPPING, MODEL_DIR / "cluster_mapping.pkl")
joblib.dump(ood_threshold, MODEL_DIR / "gmm_ood_threshold.pkl")
joblib.dump(input_limits, MODEL_DIR / "gmm_input_limits.pkl")

print("Đã lưu model GMM và kết quả phân nhóm.")